## Day 13 — End-to-End Architecture Design

**Three tasks today:**
1. Draw architecture diagram *(see PNG file)*
2. Document the full pipeline flow
3. Define the retraining strategy

This notebook documents and validates the full pipeline we built over 13 days — no new tables, just introspection and documentation.

In [0]:
from pyspark.sql.functions import col, count, avg, round as spark_round, max as spark_max, min as spark_min

CATALOG = 'ecommerce'
print('Ready')


### Task 2 — Document the Pipeline: inspect every layer we built

In [0]:
# Bronze — raw ingestion, no transformations
bronze = spark.table(f'{CATALOG}.bronze.events_raw')
print('=== BRONZE LAYER ===')
print(f'Rows : {bronze.count():,}')
print(f'Cols : {len(bronze.columns)}')
print()
bronze.printSchema()
spark.sql(f'DESCRIBE DETAIL {CATALOG}.bronze.events_raw').select('format','numFiles','sizeInBytes','partitionColumns').show(truncate=False)


In [0]:
# Silver — cleaned and validated
silver = spark.table(f'{CATALOG}.silver.events_si')
print('=== SILVER LAYER ===')
print(f'Rows : {silver.count():,}')
print(f'Cols : {len(silver.columns)}')
print()
silver.groupBy('event_type').agg(count('*').alias('events'), spark_round(avg('price'),2).alias('avg_price')).orderBy('events').show()
spark.sql(f'DESCRIBE DETAIL {CATALOG}.silver.events_si').select('format','numFiles','sizeInBytes','partitionColumns').show(truncate=False)


In [0]:
# Gold Features — aggregated user features for ML
gold_feat = spark.table(f'{CATALOG}.gold.user_features')
print('=== GOLD FEATURES ===')
print(f'Users : {gold_feat.count():,}')
print(f'Cols  : {len(gold_feat.columns)} — {gold_feat.columns}')
print()
gold_feat.describe().show()


In [0]:
# Gold Predictions — batch inference output
preds = spark.table(f'{CATALOG}.gold.user_predictions')
print('=== GOLD PREDICTIONS ===')
print(f'Total scored users : {preds.count():,}')
print()
preds.groupBy('predicted_buyer').count().show()
preds.groupBy('model_version').agg(
    count('*').alias('users'),
    spark_round(avg('purchase_probability'),4).alias('avg_prob')
).show()


In [0]:
# Gold Recommendations — ALS output
try:
    recs = spark.table(f'{CATALOG}.gold.user_recommendations')
    print('=== GOLD RECOMMENDATIONS ===')
    print(f'Recommendation rows : {recs.count():,}')
    recs.printSchema()
    display(recs.limit(5))
except Exception as e:
    print(f'user_recommendations not found: {e}')
    print('This is expected if Day 9 hit compute limits')


### Pipeline summary — row counts across all layers

In [0]:
# Full pipeline audit — row counts and sizes across all layers
tables = [
    (f'{CATALOG}.bronze.events_raw',        'Bronze',  'Raw events'),
    (f'{CATALOG}.silver.events_si',          'Silver',  'Cleaned events'),
    (f'{CATALOG}.gold.user_features',        'Gold',    'User features'),
    (f'{CATALOG}.gold.user_predictions',     'Gold',    'ML predictions'),
]

print(f'{"Table":<45} {"Layer":<10} {"Rows":>15} {"Description"}')
print('-' * 90)
for tbl, layer, desc in tables:
    try:
        n = spark.table(tbl).count()
        print(f'{tbl:<45} {layer:<10} {n:>15,} {desc}')
    except:
        print(f'{tbl:<45} {layer:<10} {"NOT FOUND":>15} {desc}')


### Task 3 — Retraining Strategy

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
MODEL_NAME = f'{CATALOG}.gold.purchase_model'

try:
    # Get registered model versions
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    print(f'=== MODEL REGISTRY: {MODEL_NAME} ===')
    print(f'Total versions: {len(versions)}')
    print()
    for v in versions:
        print(f'Version {v.version} | Status: {v.status} | Run: {v.run_id[:8]}...')
except Exception as e:
    print(f'Model not found in registry: {e}')
    print('Check model name matches what was registered in Day 7')


In [0]:
# Document the retraining strategy as code
# This is the operational definition — what a scheduler would trigger

retraining_strategy = {
    'triggers': [
        {'type': 'schedule',   'condition': 'Weekly, Monday 02:00 UTC'},
        {'type': 'drift',      'condition': 'AUC on held-out set drops below 0.92'},
        {'type': 'data_vol',   'condition': 'New events > 10% of training set size'},
        {'type': 'dist_shift', 'condition': 'Purchase rate changes by > 15% over 7 days'},
    ],
    'pipeline_steps': [
        'Day 5 — Rebuild ML dataset from latest Gold features',
        'Day 6 — Retrain Random Forest with latest data',
        'Day 7 — Log to MLflow, register new version',
        'Day 8 — Run batch inference, write new Gold predictions',
    ],
    'validation': {
        'primary_metric':   'AUC on held-out test set',
        'secondary_metric': 'Recall on predicted_buyer = True',
        'promotion_rule':   'New AUC > current production AUC + 0.005',
        'ab_test':          '10% traffic split, 7-day window',
    },
    'promotion': {
        'register':  'mlflow.register_model() with new version',
        'transition': 'Move to Production stage in MLflow registry',
        'archive':   'Move old version to Archived — never delete',
        'rollback':  'RESTORE TABLE TO VERSION AS OF N (Day 11)',
    }
}

import json
print('=== RETRAINING STRATEGY ===')
print(json.dumps(retraining_strategy, indent=2))


In [0]:
# Simulate a drift check — compare recent purchase rate vs training baseline
# In production this would run as a scheduled job

from pyspark.sql.functions import col, count, when

silver = spark.table(f'{CATALOG}.silver.events_si')

# Overall purchase rate
total  = silver.count()
buys   = silver.filter(col('event_type') == 'purchase').count()
rate   = round(buys / total * 100, 2)

# Baseline from training (Day 5)
BASELINE_PURCHASE_RATE = 1.51  # % from Day 5 feature engineering
DRIFT_THRESHOLD = 15           # % relative change triggers retraining

pct_change = abs(rate - BASELINE_PURCHASE_RATE) / BASELINE_PURCHASE_RATE * 100

print('=== DRIFT MONITOR ===')
print(f'Current purchase rate : {rate}%')
print(f'Baseline purchase rate: {BASELINE_PURCHASE_RATE}%')
print(f'Relative change       : {round(pct_change,1)}%')
print(f'Drift threshold       : {DRIFT_THRESHOLD}%')
print()
if pct_change > DRIFT_THRESHOLD:
    print('⚠️  DRIFT DETECTED — retraining should be triggered')
else:
    print('✅ No drift detected — model is still valid')


## Day 13 Complete!

| Cell | What it does |
|------|--------------|
| 2–4 | **Task 2** — Inspect Bronze, Silver, Gold layers |
| 5–6 | Inspect ML output tables (predictions + recommendations) |
| 7 | Full pipeline row count summary |
| 8 | Check MLflow model registry |
| 9 | **Task 3** — Define retraining strategy as structured doc |
| 10 | Drift monitor — check if retraining should be triggered |

**Task 1 — Architecture Diagram:** See the PNG file delivered alongside this notebook.

**Full 13-day pipeline recap:**
Raw events (109M) → Bronze → Silver → Gold Features (5.7M users) → RF Model (AUC 0.9657) → Batch Inference (Recall 92.6%) → Gold Predictions